# Lab 4 — Section D: K-th Smallest Pairwise Sum

**Task:** Find the K-th smallest sum formed by adding values at two different positions in a list.

The first approach builds and sorts every pair sum. The second sorts the input and combines a two-pointer counting routine with binary search.

In [ ]:
import random
from time import perf_counter


def kth_pair_sum_direct(values, k):
    """Build all sums from distinct positions, sort them, and select rank k."""
    all_sums = []

    for left_index in range(len(values)):
        for right_index in range(left_index + 1, len(values)):
            all_sums.append(values[left_index] + values[right_index])

    all_sums.sort()
    if not 1 <= k <= len(all_sums):
        raise ValueError("k must be between 1 and the number of distinct-position pairs")

    return all_sums[k - 1]


def count_pair_sums_at_most(sorted_values, ceiling):
    """Count distinct-index pairs whose sum is no greater than ceiling."""
    left_index = 0
    right_index = len(sorted_values) - 1
    pair_count = 0

    while left_index < right_index:
        if sorted_values[left_index] + sorted_values[right_index] <= ceiling:
            # Every position between left_index+1 and right_index also works.
            pair_count += right_index - left_index
            left_index += 1
        else:
            right_index -= 1

    return pair_count


def kth_pair_sum_binary(values, k):
    """Find the k-th pair sum without storing all pair sums."""
    if len(values) < 2:
        raise ValueError("At least two values are required")

    sorted_values = sorted(values)
    total_pairs = len(sorted_values) * (len(sorted_values) - 1) // 2
    if not 1 <= k <= total_pairs:
        raise ValueError("k is outside the valid pair-rank range")

    low = sorted_values[0] + sorted_values[1]
    high = sorted_values[-2] + sorted_values[-1]

    while low < high:
        midpoint = (low + high) // 2
        if count_pair_sums_at_most(sorted_values, midpoint) >= k:
            high = midpoint
        else:
            low = midpoint + 1

    return low


In [ ]:
random.seed(42)
sample_size = 5_000
sample_values = [random.randint(1, 100_000) for _ in range(sample_size)]
rank = 1_000_000

print("Direct approach")
started = perf_counter()
direct_result = kth_pair_sum_direct(sample_values, rank)
direct_elapsed = perf_counter() - started
print(f"Answer: {direct_result:,}")
print(f"Elapsed: {direct_elapsed:.4f} seconds")

print("\\nBinary-search approach")
started = perf_counter()
binary_result = kth_pair_sum_binary(sample_values, rank)
binary_elapsed = perf_counter() - started
print(f"Answer: {binary_result:,}")
print(f"Elapsed: {binary_elapsed:.4f} seconds")

print("\\nBoth approaches return the same value:", direct_result == binary_result)


## Explanation and Observation

**Step 1 — Direct approach**

The direct method creates a sum for every pair of different positions, sorts all those sums, and selects rank K. With 5,000 input values, it creates 12,497,500 pair sums, which requires substantial time and memory.

**Step 2 — Binary-search idea**

Instead of generating every sum, we can ask how many pair sums are less than or equal to a chosen value. This count tells us whether the K-th sum is at or below that value.

**Step 3 — Two pointers**

After sorting the input, one pointer starts at the left and another at the right. If their sum is within the limit, all pairs formed by the left value and positions up to the right pointer also fit, so they can be counted together. Otherwise, the right pointer moves left.

**Step 4 — Search for the answer**

Binary search repeatedly checks the midpoint between the smallest and largest possible pair sums. The pair count determines which half can contain the required rank.

**Q10. What made the faster method quicker?**

It avoids creating and sorting millions of pair sums. Instead, it uses two pointers to count qualifying pairs and binary search to locate the required sum. The timing cell prints the measurements for the current machine.